# melakukan data cleaning 

melihat info data

In [4]:
import pandas as pd
import numpy as np
import ast
import io

df = pd.read_csv('Top_250_movies_on_imdb_in_2026.csv')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   250 non-null    object 
 1   url                  250 non-null    object 
 2   primaryTitle         250 non-null    object 
 3   originalTitle        250 non-null    object 
 4   type                 250 non-null    object 
 5   description          250 non-null    object 
 6   primaryImage         250 non-null    object 
 7   thumbnails           250 non-null    object 
 8   trailer              243 non-null    object 
 9   contentRating        244 non-null    object 
 10  startYear            250 non-null    int64  
 11  endYear              0 non-null      float64
 12  releaseDate          249 non-null    object 
 13  interests            250 non-null    object 
 14  countriesOfOrigin    250 non-null    object 
 15  externalLinks        250 non-null    obj

melihat total data yang  kosong

In [5]:
df.isna().sum()

id                       0
url                      0
primaryTitle             0
originalTitle            0
type                     0
description              0
primaryImage             0
thumbnails               0
trailer                  7
contentRating            6
startYear                0
endYear                250
releaseDate              1
interests                0
countriesOfOrigin        0
externalLinks            0
spokenLanguages          0
filmingLocations         0
productionCompanies      0
budget                  24
grossWorldwide           4
genres                   0
isAdult                  0
runtimeMinutes           0
averageRating            0
numVotes                 0
metascore               18
dtype: int64

hasil yang didapatkan pada pengecekan data kali ini adalah bahwa terdapat beberapa baris data hilang atau kosong pada beberapa kolom yaitu endYear, budget, dan metascore. oleh karena itu baris yang harus segera dihapus terutama pada kolom endyear yang memang tidak sesuai dengan data pada sebuah film karena bukan series/tv.

menghapus kolom endyear dan mengubah nama starYear menjadi Year

In [6]:
df.drop(columns=['endYear'], errors='ignore', inplace=True)
df.rename(columns={'startYear': 'Year'}, inplace = True)
df.columns

Index(['id', 'url', 'primaryTitle', 'originalTitle', 'type', 'description',
       'primaryImage', 'thumbnails', 'trailer', 'contentRating', 'Year',
       'releaseDate', 'interests', 'countriesOfOrigin', 'externalLinks',
       'spokenLanguages', 'filmingLocations', 'productionCompanies', 'budget',
       'grossWorldwide', 'genres', 'isAdult', 'runtimeMinutes',
       'averageRating', 'numVotes', 'metascore'],
      dtype='object')

lalu untuk bagian budget dan grossworldwide diisi dengan angka nol sebagai catatan bahwa budget tidak diketahui dan untuk metascore diisi dengan nilai rata-rata atau median agar tidak merusak perhitungan. untuk sisanya seperti trailer, content rating, releasedate diisi dengan unknown

In [7]:
df['budget'] = df['budget'].fillna(0)
df['grossWorldwide'] = df['grossWorldwide'].fillna(0)
df['metascore']=df['metascore'].fillna(df['metascore'].median())
df['trailer'] = df['trailer'].fillna('unknown')
df['releaseDate'] = df['releaseDate'].fillna('unknown')
df['contentRating'] = df['contentRating'].fillna('unknown')

df.isna().sum()

id                     0
url                    0
primaryTitle           0
originalTitle          0
type                   0
description            0
primaryImage           0
thumbnails             0
trailer                0
contentRating          0
Year                   0
releaseDate            0
interests              0
countriesOfOrigin      0
externalLinks          0
spokenLanguages        0
filmingLocations       0
productionCompanies    0
budget                 0
grossWorldwide         0
genres                 0
isAdult                0
runtimeMinutes         0
averageRating          0
numVotes               0
metascore              0
dtype: int64

In [8]:
print(df)

            id                                    url  \
0    tt0111161  https://www.imdb.com/title/tt0111161/   
1    tt0068646  https://www.imdb.com/title/tt0068646/   
2    tt0468569  https://www.imdb.com/title/tt0468569/   
3    tt0071562  https://www.imdb.com/title/tt0071562/   
4    tt0167260  https://www.imdb.com/title/tt0167260/   
..         ...                                    ...   
245  tt0032138  https://www.imdb.com/title/tt0032138/   
246  tt0058946  https://www.imdb.com/title/tt0058946/   
247  tt0476735  https://www.imdb.com/title/tt0476735/   
248  tt0019254  https://www.imdb.com/title/tt0019254/   
249  tt4016934  https://www.imdb.com/title/tt4016934/   

                                      primaryTitle  \
0                         The Shawshank Redemption   
1                                    The Godfather   
2                                  The Dark Knight   
3                            The Godfather Part II   
4    The Lord of the Rings: The Return of the

untuk kolom seperti genres, countriOfOrigin, spokenLanguages menggunakan format string list['US'], dan membersihkan tanda[,],', dan " agar menjadi teks bersih biasa

In [9]:
columns_to_clean = ['genres', 'countryOfOrigin', 'spokenLanguages', 'interests', 'externalLinks', 'filmingLocations']

for col in columns_to_clean:
    if col in df.columns:
        df[col]=df[col].astype(str)\
        .str.replace(r"[\[\]\'\"]","", regex = True)\
        .str.strip()

        df[col] = df[col].replace('nan',np.nan)

membersihkan data JSON/Dict yang berantakan di'productionCompanies' dan 'thumbnails'

In [10]:
def clean_production_companies(val):
    if pd.isna(val) or val == '[]': return np.nan
    try:
        extracted = eval(val)
        if isinstance(extracted, list) and len(extracted) > 0:
            return extracted[0].get('name', np.nan)
    except:
        pass
    # Backup plan jika eval gagal karena tanda kutip berantakan
    return str(val).split("'name': '")[-1].split("'")[0] if "name" in str(val) else val

df['productionCompanies'] = df['productionCompanies'].apply(clean_production_companies)


In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   250 non-null    object 
 1   url                  250 non-null    object 
 2   primaryTitle         250 non-null    object 
 3   originalTitle        250 non-null    object 
 4   type                 250 non-null    object 
 5   description          250 non-null    object 
 6   primaryImage         250 non-null    object 
 7   thumbnails           250 non-null    object 
 8   trailer              250 non-null    object 
 9   contentRating        250 non-null    object 
 10  Year                 250 non-null    int64  
 11  releaseDate          250 non-null    object 
 12  interests            250 non-null    object 
 13  countriesOfOrigin    250 non-null    object 
 14  externalLinks        250 non-null    object 
 15  spokenLanguages      250 non-null    obj

In [12]:
df.to_csv('data_top_movies_cleaned.csv', index=False)